In [ ]:
from datasets import load_dataset, Dataset
import requests
import json
import pathlib
from collections import Counter
import re

We will now load all SmolDoc datasets from the `google/smol` dataset on Hugging Face Datasets, and cross-reference them with our factuality annotations.

In [ ]:
dataset="google/smol"
def list_smoldoc_configs(dataset="google/smol"):
    url = f"https://datasets-server.huggingface.co/splits?dataset={dataset}"
    r = requests.get(url); r.raise_for_status()
    data = r.json()
    configs = sorted({item["config"] for item in data["splits"]})
    return [c for c in configs if c.lower().startswith("smoldoc__")]

In [ ]:
smoldoc_configs = list_smoldoc_configs()
print(f"{len(smoldoc_configs)} SmolDoc configs")

We load all the SmolDoc configs into a dictionary of datasets.

In [ ]:
datasets_dict = {}
for cfg in smoldoc_configs:
    print("Loading config:", cfg)
    ds = load_dataset(dataset, cfg)
    datasets_dict[cfg] = ds

Simple statistics: how many unique topic ids are there across all SmolDoc datasets?

In [ ]:
# find all unique topic ids across all langauges
all_topic_ids = set()
for cfg, ds in datasets_dict.items():
    topic_ids = set(ds['train']['id'])
    all_topic_ids.update(topic_ids)
print(f"Total unique topic ids across all SmolDoc topics: {len(all_topic_ids)}")

In [ ]:
datasets_dict['smoldoc__en_bgq']['train']

Now load the factuality annotations from JSON, and load it as a Dataset. We will then compare the annotated topic ids with those present in the English SmolDoc datasets.

In [ ]:
from data_utils import load_topic_ratings
JSON_PATH = "smoldoc-factuality-ratings.json"  # adjust if needed


topic_ds = load_topic_ratings(JSON_PATH)
topic_ds

We have found out that there are not only English as source language datasets in SmolDoc, so we will only compare with those, to be consistent. Let us find all English SmolDoc configs, and collect their topic ids.

In [ ]:
english_cfgs = [c for c in datasets_dict if c.startswith("smoldoc__en_")]
english_topic_ids = set().union(*[set(datasets_dict[c]["train"]["id"]) for c in english_cfgs])

annot_topic_ids = set(x for x in topic_ds["topic_key"] if x is not None)

missing_in_annotations = sorted(english_topic_ids - annot_topic_ids)
extra_in_annotations_vs_english = sorted(annot_topic_ids - english_topic_ids)

print(f"# Annotation rows: {len(topic_ds)}")
print(f"# Unique annotated topic_ids: {len(annot_topic_ids)}")
print(f"# English topic_ids in datasets: {len(english_topic_ids)}")
print(f"# English topic_ids missing annotations: {len(missing_in_annotations)}")
print(f"# Annotated topic_ids not present in English: {len(extra_in_annotations_vs_english)}")

Now we will join the factuality annotations to all SmolDoc datasets (all languages), for those topic ids that are present in the annotations. This will allow us to analyze factuality across all languages.

In [ ]:
keep_cols = [
    'annotator_1_label', 'annotator_1_notes',
    'annotator_2_label', 'annotator_2_notes',
    'annotator_3_label', 'annotator_3_notes'
]

# Build lookup dict: topic_key -> annotations
annot_fields_by_id = {
    k: {col: v for col, v in zip(keep_cols, vals)}
    for k, *vals in zip(
        topic_ds["topic_key"],
        *[topic_ds[col] for col in keep_cols]
    )
}

def _join_annotations(batch):
    ids = batch["id"]
    # Vectorized per-column build
    return {
        col: [annot_fields_by_id[i][col] if i in annot_fields_by_id else None for i in ids]
        for col in keep_cols
    }

# Join all configs
fact_annot_ds = {
    cfg: dsdict["train"].map(_join_annotations, batched=True)
    for cfg, dsdict in datasets_dict.items()
}

Here is an example of one of the joined datasets with factuality annotations.

In [ ]:
example_cfg = 'smoldoc__en_sw'
fact_annot_ds[example_cfg]